# PS 8 — Breast Cancer Wisconsin Diagnostic (UCI / sklearn)

**DWM practical | Run cells top-to-bottom.**

All datasets are read from `../data/` and results saved in `../outputs/`. Run `python download_datasets.py` in the repo root beforehand.

**Goal:** load the authentic UCI Wisconsin Diagnostic dataset through sklearn, check quality and numeric features, encode diagnosis as malignant=1 and benign=0, then Min-Max normalize predictor columns (not diagnosis).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
DATA = Path('../data')
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)

from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()
# sklearn target: 0=malignant, 1=benign. Convert explicitly into M/B strings then re-encode.
df = df.rename(columns={'target':'diagnosis'})
df['diagnosis'] = df['diagnosis'].map({0:'M',1:'B'})
before = df.copy()

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:'); print(df.dtypes)
print('Missing values:'); print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())
display(df.head())
display(df.describe(include='all').T)

In [ ]:
print('Categorical columns:',df.select_dtypes(include=['object','string','category']).columns.tolist())
print('Numeric columns:',df.select_dtypes(include='number').columns.tolist())
df = df.drop_duplicates().copy()
features = df.columns.drop('diagnosis').tolist()
for c in features:
    df[c] = pd.to_numeric(df[c],errors='coerce')
    df[c] = df[c].fillna(df[c].median())
df['diagnosis'] = df['diagnosis'].astype('string').str.upper().str.strip()
print('Diagnosis counts:',df['diagnosis'].value_counts().to_dict())
df['diagnosis_encoded'] = df['diagnosis'].map({'B':0,'M':1}).astype(int)

**Encoding:** B → 0, M → 1. **Scaling:** Min-Max transforms the 30 numeric measured features into [0,1]; diagnosis target remains binary.

In [ ]:
print('Before normalization:');display(df[features[:5]].head())
df[features] = MinMaxScaler().fit_transform(df[features])
print('After normalization:');display(df[features[:5]].head())
processed = df.drop(columns=['diagnosis']).copy()
print('Before:');display(before.head())
print('After:');display(processed.head())
print('Final shape:',processed.shape)
print('Any nulls:',processed.isna().sum().sum())
processed.to_csv(OUT/'PS08_breast_cancer_processed.csv',index=False)